In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

In [2]:
dtype = {
    # category
    "transaction_type": "category",
    "merchant_category": "category",
    "location": "category",
    "device_used": "category",
    "fraud_type": "category",
    "payment_channel": "category",
    "ip_address": "category",
    "sender_persona": "category",
    "user_top_category": "category",
    "ip_geo_region": "category",

    # string
    "transaction_id": "string",
    "device_hash": "string",

    # uint64
    "sender_account": "uint64",
    "receiver_account": "uint64",

    # uint8
    "is_fraud": "uint8",
    "velocity_score": "uint8",
    "bvn_linked": "uint8",
    "new_device_transaction": "uint8",
    "geospatial_velocity_anomaly": "uint8",
    "txn_hour": "uint8",
    "is_weekend": "uint8",
    "is_salary_week": "uint8",
    "is_night_txn": "uint8",
    "device_seen_count": "uint8",
    "is_device_shared": "uint8",
    "is_ip_shared": "uint8",
    "user_txn_count_total": "uint8",
    "user_txn_frequency_24h": "uint8",
    "txn_count_last_1h": "uint8",
    "txn_count_last_24h": "uint8",

    # uint16
    "ip_seen_count": "uint16",

    # float32
    "time_since_last_transaction": "float32",
    "spending_deviation_score": "float32",
    "geo_anomaly_score": "float32",
    "amount_ngn": "float32",
    "user_avg_txn_amt": "float32",
    "user_std_txn_amt": "float32",
    "total_amount_last_1h": "float32",
    "time_since_last": "float32",
    "avg_gap_between_txns": "float32",
    "merchant_fraud_rate": "float32",
    "channel_risk_score": "float32",
    "persona_fraud_risk": "float32",
    "location_fraud_risk": "float32",
}

In [3]:
df = pd.read_csv(
    'data/transactions.csv',
    dtype=dtype,
    parse_dates=["timestamp"],
)
df.head()

,transaction_id,timestamp,sender_account,receiver_account,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,...,txn_count_last_1h,txn_count_last_24h,total_amount_last_1h,time_since_last,avg_gap_between_txns,merchant_fraud_rate,channel_risk_score,persona_fraud_risk,location_fraud_risk,ip_geo_region
0,T2162315,2023-01-24 09:54:06.198396,1000018177,8385560081,deposit,Local Market Purchase,Aba,mobile,0,NaN,...,1,1,654135.0625,0.000000,0.000000,0.1,0.8,0.5,0.1,South East
1,T1764581,2023-02-22 16:16:19.271951,1000018177,5643014197,payment,SPAR Purchase,Onitsha,mobile,0,NaN,...,2,2,687449.1250,42142.218750,21071.109375,0.1,0.3,0.5,0.1,South East
2,T3305551,2023-05-04 16:01:42.312142,1000018177,7722691989,withdrawal,Other Transaction,Onitsha,web,0,NaN,...,3,3,719985.7500,102225.382812,48122.535156,0.1,0.8,0.5,0.0,South East
3,T174955,2023-05-07 13:15:03.037215,1000018177,4987435115,payment,Ikeja Electric Bill,Benin City,atm,0,NaN,...,4,4,733430.8750,4153.345215,37130.238281,0.1,0.3,0.5,0.1,South South
4,T3695059,2023-06-08 11:37:39.155188,1000018177,7939643449,withdrawal,Arik Air Flight,Aba,web,0,NaN,...,5,5,858543.4375,45982.601562,38900.710938,0.1,0.6,0.5,0.0,South East


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 45 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   transaction_id               string        
 1   timestamp                    datetime64[us]
 2   sender_account               uint64        
 3   receiver_account             uint64        
 4   transaction_type             category      
 5   merchant_category            category      
 6   location                     category      
 7   device_used                  category      
 8   is_fraud                     uint8         
 9   fraud_type                   category      
 10  time_since_last_transaction  float32       
 11  spending_deviation_score     float32       
 12  velocity_score               uint8         
 13  geo_anomaly_score            float32       
 14  payment_channel              category      
 15  ip_address                   category      
 16  device_hash

In [6]:
df.isnull().sum()

transaction_id                       0
timestamp                            0
sender_account                       0
receiver_account                     0
transaction_type                     0
merchant_category                    0
location                             0
device_used                          0
is_fraud                             0
fraud_type                     4820447
time_since_last_transaction     896513
spending_deviation_score             0
velocity_score                       0
geo_anomaly_score                    0
payment_channel                      0
ip_address                           0
device_hash                          0
amount_ngn                           0
bvn_linked                           0
new_device_transaction               0
sender_persona                       0
geospatial_velocity_anomaly          0
txn_hour                             0
is_weekend                           0
is_salary_week                       0
is_night_txn             

### Overview data

In [7]:
overview = {
    "Number of transactions": len(df),
    "Number of features": df.shape[1],
    "Numerical features": len(df.select_dtypes(include=["number"]).columns),
    "Categorical features": len(df.select_dtypes(include=["category", "object", "string"]).columns),
    "Fraud transactions": df["is_fraud"].sum(),
    "Non-fraud transactions": (df["is_fraud"] == 0).sum(),
    "Fraud ratio (%)": round(df["is_fraud"].mean()*100,2),
    "Fraud types": df["fraud_type"].nunique()
}

pd.DataFrame(overview.items(), columns=["Metric","Value"])

,Metric,Value
0,Number of transactions,5000000
1,Number of features,45
2,Numerical features,32
3,Categorical features,12
4,Fraud transactions,179553
5,Non-fraud transactions,4820447
6,Fraud ratio (%),3.59
7,Fraud types,3


In [8]:
core_numeric = [
    "amount_ngn",
    "velocity_score",
    "geo_anomaly_score",
    "merchant_fraud_rate",
    "channel_risk_score",
    "persona_fraud_risk",
    "location_fraud_risk",
    "time_since_last_transaction",
    "txn_count_last_1h",
    "txn_count_last_24h"
]

df[core_numeric].describe().T

,count,mean,std,min,25%,50%,75%,max
amount_ngn,5000000.0,748292.437500,1.545519e+06,22.809999,33209.951172,163810.023438,695490.796875,2.223067e+07
velocity_score,5000000.0,10.501320,5.766842e+00,1.000000,5.000000,11.000000,16.000000,2.000000e+01
geo_anomaly_score,5000000.0,0.500029,2.886349e-01,0.000000,0.250000,0.500000,0.750000,1.000000e+00
merchant_fraud_rate,5000000.0,0.036070,1.460525e-03,0.000000,0.035687,0.036051,0.036406,1.000000e+00
channel_risk_score,5000000.0,0.524975,1.920316e-01,0.300000,0.300000,0.600000,0.600000,8.000000e-01
persona_fraud_risk,5000000.0,0.533270,1.246131e-01,0.400000,0.400000,0.500000,0.700000,7.000000e-01
location_fraud_risk,5000000.0,0.036070,1.197160e-03,0.000000,0.035715,0.036026,0.036316,1.000000e+00
time_since_last_transaction,4103487.0,1.525799,3.576569e+03,-8777.814453,-2562.375977,0.844275,2568.338867,8.757759e+03
txn_count_last_1h,5000000.0,3.778863,2.315144e+00,1.000000,2.000000,3.000000,5.000000,2.300000e+01
txn_count_last_24h,5000000.0,3.778863,2.315144e+00,1.000000,2.000000,3.000000,5.000000,2.300000e+01


In [9]:
categorical_cols = [
    "transaction_type",
    "merchant_category",
    "payment_channel",
    "sender_persona",
    "location"
]

summary = []

for col in categorical_cols:
    vc = df[col].value_counts()
    summary.append({
        "Feature": col,
        "Unique": df[col].nunique(),
        "Most Frequent": vc.index[0],
        "Count": vc.iloc[0]
    })

pd.DataFrame(summary)

,Feature,Unique,Most Frequent,Count
0,transaction_type,4,deposit,1250593
1,merchant_category,21,Other Transaction,1251802
2,payment_channel,4,Bank Transfer,1251164
3,sender_persona,3,Student,1672920
4,location,10,Kaduna,501261


In [10]:
missing = (
    df.isna()
      .sum()
      .to_frame("Missing")
)

missing["Percentage"] = (
    missing["Missing"]/len(df)*100
).round(2)

missing = missing[missing["Missing"]>0]

missing

,Missing,Percentage
fraud_type,4820447,96.41
time_since_last_transaction,896513,17.93
